# PDSI — Checkpoint 1 | Preparação do dataset histórico

**Pessoa 1:** preparação, documentação e inspeção das imagens. Este notebook **não altera os originais**.

**Entrada:** envie `imagens_historicas_originais.zip` e `catalogo_dataset_pdsi.csv` gerados para o projeto.

**Saída:** imagens padronizadas preservando proporção, relatório técnico CSV, contato visual e ZIP pronto para compartilhar.

> **Atenção:** resolução em pixels não garante detalhes microscópicos; o dataset não possui rótulos/máscaras de verdade-terreno. As avaliações quantitativas de segmentação dependerão de anotações adicionais.

In [ ]:
import io, zipfile, shutil, csv, hashlib
from pathlib import Path
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from google.colab import files

BASE = Path('/content/pdsi')
ORIG = BASE / 'originais'
PROC = BASE / 'padronizadas'
RESULT = BASE / 'resultados'
for pasta in [ORIG, PROC, RESULT]:
    pasta.mkdir(parents=True, exist_ok=True)
print('Pastas prontas:', BASE)

## 1. Enviar arquivos e extrair imagens

In [ ]:
enviados = files.upload()
for nome, dados in enviados.items():
    if nome.lower().endswith('.zip'):
        with zipfile.ZipFile(io.BytesIO(dados)) as z:
            for membro in z.infolist():
                caminho = Path(membro.filename)
                if membro.is_dir() or caminho.suffix.lower() not in {'.jpg','.jpeg','.png','.tif','.tiff'}:
                    continue
                # Extrair com caminho seguro e sem subpastas
                destino = ORIG / caminho.name
                destino.write_bytes(z.read(membro))
    elif nome.lower().endswith('.csv'):
        (BASE/'catalogo_dataset_pdsi.csv').write_bytes(dados)
    elif Path(nome).suffix.lower() in {'.jpg','.jpeg','.png','.tif','.tiff'}:
        (ORIG / Path(nome).name).write_bytes(dados)

arquivos = sorted(p for p in ORIG.iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png','.tif','.tiff'})
print(f'{len(arquivos)} imagens carregadas')
assert arquivos, 'Envie o ZIP fornecido ou imagens individuais.'


## 2. Carregar metadados e conferir a integridade

In [ ]:
catalogo = BASE/'catalogo_dataset_pdsi.csv'
if catalogo.exists():
    df_meta = pd.read_csv(catalogo, encoding='utf-8-sig').fillna('')
else:
    df_meta = pd.DataFrame({'arquivo_original':[p.name for p in arquivos]})

registros = []
for p in arquivos:
    try:
        with Image.open(p) as im:
            w,h = im.size; modo=im.mode; formato=im.format
        registros.append({'arquivo_original':p.name,'largura_real_px':w,'altura_real_px':h,'modo_real':modo,'formato_real':formato,'mb_real':round(p.stat().st_size/1024**2,3),'sha256_real':hashlib.sha256(p.read_bytes()).hexdigest(),'status':'OK'})
    except Exception as e:
        registros.append({'arquivo_original':p.name,'status':f'ERRO: {e}'})
inspecao=pd.DataFrame(registros)
relatorio=df_meta.merge(inspecao,on='arquivo_original',how='outer')
if 'sha256' in relatorio.columns:
    relatorio['hash_confere'] = relatorio.apply(lambda r: str(r.get('sha256','')) == str(r.get('sha256_real','')) if r.get('sha256','') else 'Sem referência',axis=1)
display(relatorio[['arquivo_original','largura_real_px','altura_real_px','modo_real','status']])
relatorio.to_csv(RESULT/'relatorio_inspecao.csv',index=False,encoding='utf-8-sig')

## 3. Padronizar com proporção preservada

Utilizamos até **1024 pixels no lado maior**, somente reduzindo imagens maiores; **não fazemos upscale** de arquivos pequenos. As dimensões originais ficam preservadas nos metadados. Valores RGB, JPEG qualidade 95.

In [ ]:
LADO_MAXIMO = 1024
infos=[]
for i,p in enumerate(arquivos,1):
    with Image.open(p) as im:
        rgb=im.convert('RGB')
        w,h=rgb.size
        escala=min(1.0,LADO_MAXIMO/max(w,h))
        novo=(max(1,round(w*escala)),max(1,round(h*escala)))
        if novo != (w,h):
            rgb=rgb.resize(novo,Image.Resampling.LANCZOS)
        arquivo_saida=f'img_{i:03d}.png'
        rgb.save(PROC/arquivo_saida, format='PNG', optimize=True)
        infos.append({'arquivo_original':p.name,'arquivo_padronizado':arquivo_saida,'largura_final':novo[0],'altura_final':novo[1],'escala':escala})

mapa=pd.DataFrame(infos)
mapa.to_csv(RESULT/'mapa_arquivos.csv', index=False,encoding='utf-8-sig')
display(mapa)

## 4. Visualizar todas as imagens (folha de contato)

In [ ]:
n=len(infos);cols=3;linhas=int(np.ceil(n/cols))
fig,axs=plt.subplots(linhas,cols,figsize=(14,4.4*linhas))
for ax in np.ravel(axs): ax.axis('off')
for ax,linha in zip(np.ravel(axs),infos):
    img=np.asarray(Image.open(PROC/linha['arquivo_padronizado']))
    ax.imshow(img); ax.set_title(linha['arquivo_padronizado']+' — '+linha['arquivo_original'],fontsize=8)
plt.tight_layout()
plt.savefig(RESULT/'folha_de_contato.png',dpi=150,bbox_inches='tight')
plt.show()

## 5. Inspeção simples de qualidade (sem IA)

Média e desvio da escala de cinza; **variância do Laplaciano** como indicador de nitidez aparente. Não é uma nota objetiva de qualidade e valores não devem ser comparados entre diferentes tipos de arte sem cautela.

In [ ]:
qualidade=[]
for linha in infos:
    img=cv2.imread(str(PROC/linha['arquivo_padronizado']))
    cinza=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    qualidade.append({'arquivo':linha['arquivo_padronizado'],'brilho_medio':round(float(cinza.mean()),2),'desvio_cinza':round(float(cinza.std()),2),'variancia_laplaciano':round(float(cv2.Laplacian(cinza,cv2.CV_64F).var()),2)})
df_qualidade=pd.DataFrame(qualidade)
display(df_qualidade)
df_qualidade.to_csv(RESULT/'qualidade_exploratoria.csv',index=False,encoding='utf-8-sig')

## 6. Selecionar uma imagem para testar histograma e preparação

In [ ]:
escolha = 0 # índices de 0 a len(infos)-1
assert 0 <= escolha < len(infos)
img = cv2.cvtColor(cv2.imread(str(PROC/infos[escolha]['arquivo_padronizado'])),cv2.COLOR_BGR2RGB)
fig,axs=plt.subplots(1,2,figsize=(13,4))
axs[0].imshow(img);axs[0].set_title(infos[escolha]['arquivo_original']);axs[0].axis('off')
for canal,cor in enumerate(['red','green','blue']):
    hist=cv2.calcHist([img],[canal],None,[256],[0,256]).ravel()
    axs[1].plot(hist / hist.sum(),color=cor,label=cor)
axs[1].set_title('Histograma RGB normalizado');axs[1].set_xlabel('Intensidade');axs[1].set_ylabel('Proporção');axs[1].legend()
plt.tight_layout();plt.show()

## 7. Exportar o dataset para o grupo

O ZIP contém as imagens padronizadas e CSVs. As imagens originais permanecem no ZIP recebido. Para GitHub, prefiram registrar a origem e a licença e não subir arquivos de terceiros sem conferir as permissões.

In [ ]:
saida=Path('/content/pdsi_preparado.zip')
with zipfile.ZipFile(saida,'w',zipfile.ZIP_DEFLATED) as z:
    for pasta in [PROC,RESULT]:
        for arq in pasta.iterdir():
            if arq.is_file():z.write(arq,arcname=f'{pasta.name}/{arq.name}')
    if catalogo.exists():z.write(catalogo,arcname='catalogo_dataset_pdsi.csv')
print(saida, round(saida.stat().st_size/1024**2,2),'MB')
files.download(str(saida))

## Antes de entregar ao grupo
- [ ] Conferir as 10 imagens na folha de contato.
- [ ] Verificar título, autoria e fonte **da cópia realmente baixada**.
- [ ] Confirmar o registro de Rama, ainda pendente.
- [ ] Registrar a licença ou condições de reprodução por arquivo.
- [ ] Escolher 3–5 imagens para anotar regiões de interesse.
- [ ] Salvar as máscaras de referência com as mesmas dimensões das imagens usadas pela Pessoa 2.

**Validação futura:** sem máscaras de referência, histogramas e filtros servem à exploração visual; não há como calcular IoU de segmentação. Para avaliar, anotem pequenas regiões manualmente ou usem dataset previamente rotulado.